# Task 2.3: transmon-free readout

**In plain words.** The bit lives in the memory. To read it we do not add a transmon: we turn on a weak drive $\varepsilon_Z(a+a^\dagger)$ and watch the buffer. The two wells send the buffer to opposite steady amplitudes, and that contrast is the measurement.

**Prepare, then drive.** $\varepsilon_Z(a+a^\dagger)$ pushes $\langle a\rangle$ along the imaginary axis. It is a $Z_L$ rotation. It cannot pick a well out of vacuum, so each run starts from $|\pm\alpha\rangle$ with $\varepsilon_Z=0$ for 3 µs, and only then is the drive switched on.

**Why $\langle b\rangle$ rings.** Linearize $a = s\alpha + \delta a$ about either well. The buffer amplitude and the memory displacement are a 2×2 system whose characteristic equation is $\lambda^2 + (\kappa_b/2)\lambda + 4 g_2^2\alpha^2 = 0$. It is underdamped because $8 g_2\alpha = 80.5 > \kappa_b = 16.3$ rad/µs (Marquet's adiabaticity-breaking line). The one-mode model, which assumes the buffer empties instantly, would settle in $1/(2\kappa_2\alpha^2) \approx 20$ ns. That assumption is false here: $g_2/\kappa_b = 0.29$.

**Answer (2×2 model, `src/linresp.py`).** $\langle b\rangle$ **oscillates**. It does not relax monotonically. The ring is at **3.138 MHz** (period 0.32 µs), quality factor **Q = 2.465**. The envelope decays at $\kappa_b/4 = 4.084$/µs, independent of $g_2$: within **5% by 0.734 µs** and within **1% by 1.128 µs**. On the same 2 ns grid the simulation uses, those thresholds are crossed at **0.682 µs** and **1.132 µs**.

**Contrast at the drive the prompt asks for,** $\varepsilon_Z = 0.1 g_2$ (well below $g_2$). Réglade S17: $\langle b\rangle_\mathrm{ss} = \mp\varepsilon_Z/(2\alpha g_2)$, so $\Delta\langle b\rangle_\mathrm{ss} = \langle b\rangle_+ - \langle b\rangle_- = -0.1/\alpha = -0.04762$. $\mathrm{SNR}(1\,\mu\mathrm{s}) = \kappa_b\int_0^1|\Delta\langle b\rangle|^2\,dt \approx 0.038$. The $\kappa_a$ offset $\langle b\rangle_0 = i\kappa_a/(4 g_2) = 0.00305i$ is common to both wells and cancels in $\Delta\langle b\rangle$.

**What the readout costs.** S18: $\Gamma_Z(\varepsilon_Z)-\Gamma_Z(0) = 2\varepsilon_Z^2/(\kappa_2\alpha^2) = (\kappa_b/2)|\Delta\langle b\rangle_\mathrm{ss}|^2$. That identity is why $\mathrm{SNR} = \kappa_b\int|\Delta\langle b\rangle|^2 dt$ is the right figure of merit. Phase information is supposed to leave during a $Z_L$ measurement. The cost that limits how hard the drive can be is $\Gamma_X$, not $\Gamma_Z$: the drive is even in $\varepsilon_Z$, and bit flips are what a memory must not do.

The cells below write the analytic record, then read the two-mode records (ring frequency, both settle times, $\Delta\Gamma_Z$) and the three figures. The full production command is commented out: commit the code first, run second, so `provenance.dirty` stays false.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))          # repo root: results/ and figures/ resolve here
from IPython.display import Image, display
from src import config as C              # double precision before any array is built
from src.results import load_record
from tasks import task_2_3

def show(name, w=980):
    display(Image(f'figures/{name}.png', width=w))


In [ ]:
# Instant: analytic 2x2 record. No mesolve.
task_2_3.main(part='linresp')
lin = load_record(task_2_3.LINRESP_ID)
print(lin['id'], lin['value'], 'MHz')
print({k: lin['numerics'][k] for k in ('Q', 'decay_per_us', 't_settle_5pct_us', 't_settle_1pct_us', 'underdamped')})


In [ ]:
# Two-mode answers. These files exist only after the production run below.
ring = load_record('t2_3.ring_freq.td.two_mode.a2.10')
t5 = load_record('t2_3.settle_5pct.td.two_mode.a2.10')
t1 = load_record('t2_3.settle_1pct.td.two_mode.a2.10')
bp = load_record('t2_3.b_ss.td.two_mode.a2.10.ez0.1g2.wellp')
bm = load_record('t2_3.b_ss.td.two_mode.a2.10.ez0.1g2.wellm')
print(f"ring {ring['value']:.4f} MHz  (linresp 3.138, rel {ring['cross_check'][0]['rel_diff']:.4f})")
print(f"settle 5% {t5['value']:.4f} us, 1% {t1['value']:.4f} us")
print(f"Re<b>+ = {bp['value']:+.5f}, Re<b>- = {bm['value']:+.5f}, "
      f"Delta b = {bp['value'] - bm['value']:+.5f}")
print(f"overshoots above 1%: {ring['numerics']['n_overshoots_1pct']}  "
      f"(oscillating if this is >= 2)")
print(f"symmetry residual {bp['numerics']['symmetry_residual']:.2e}, "
      f"Ehrenfest {bp['numerics']['ehrenfest_residual']:.2e}")
for x in (0.2, 0.5, 1.0):
    d = load_record(f't2_3.dephasing.td.two_mode.a2.10.ez{x:g}g2')
    print(f"Delta Gamma_Z at {x:g} g2: {d['value']:.4f} /us  "
          f"S18 rel {d['cross_check'][0]['rel_diff']:.3f} (tol {d['cross_check'][0]['tol']})")


In [ ]:
show('task_2_3_buffer_signals')
show('task_2_3_linearity')
show('task_2_3_dephasing')


In [ ]:
# Production (commit tasks/task_2_3.py first, then run, then commit results/ and figures/).
# Time one +alpha well before the sweeps; queue the part if that extrapolates past 15 min.
# task_2_3.main(part='signals')
# task_2_3.main(part='dephasing')
# task_2_3.main(part='figures')


**Reading the plots.** Left and centre: Re$\langle b\rangle$ for the two wells is equal and opposite, and the imaginary part sits on the shared $0.003i$ baseline. Dashed curves are the 2×2 model. Right: $\Delta\langle b\rangle/\Delta\langle b\rangle_\mathrm{ss}$ at $0.1 g_2$ crosses the 5% and 1% bands while still ringing, which is the underdamped buffer, not a slow monotonic crawl.

The linearity panel stays inside 5% of S17 for $\varepsilon_Z \le g_2$. Past that the well has moved by $\delta a/\alpha = \varepsilon_Z/(2\kappa_2\alpha^3)$ (4.6% at $g_2$) and the deviation is the large-signal correction, reported rather than tuned away. The dephasing panel is S18: $\Delta\Gamma_Z$ tracks $2\varepsilon_Z^2/(\kappa_2\alpha^2)$ and the pointer $(\kappa_b/2)|\Delta b_\mathrm{ss}|^2$ built from the measured buffer amplitude.